# Using Fourier Neural Operators (FNOs) to predict the solution of 1D Burgers' equation

This notebook uses data from 1D Burgers' equation to train Fourier Neural Operators (FNOs) and also predict the solution. The problem setup is:

- **Domain**: $\Omega = [0, 1] \text{ and } t = [0, 1]$
- **Initial Conditions**: $u(x, t = 0) = u_0$
- **Input**: $a(x) = \left[u_0(x), X\right]$
- **Output**: $u(x, t=1)$

1. Import necessary libraries

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
import matplotlib.pyplot as plt
from timeit import default_timer

2. Define parameters

In [ ]:
torch.manual_seed(0)
np.random.seed(0)

# TODO: define the parameters
ntrain = 1000
ntest = 100

sub = 2**3 #subsampling rate
h = 2**13 // sub #total grid size divided by the subsampling rate
s = h

batch_size = 20
learning_rate = 0.001

epochs = 200
step_size = 50 #lr scheduler
gamma = 0.5  #lr scheduler

modes = 16 #Fourier modes
width = 64 #num_chanells after lifting

3. Read input data

In [ ]:
fulldata = np.load('burgers.npz')
x_data = fulldata['ax'][:,::sub].astype(np.float32)
y_data = fulldata['y'][:,::sub].astype(np.float32)
# create tensors
x_train = torch.tensor(x_data[:ntrain,...], dtype=torch.float32)
y_train = torch.tensor(y_data[:ntrain,...], dtype=torch.float32)
x_test = torch.tensor(x_data[-ntest:,...], dtype=torch.float32)
y_test = torch.tensor(y_data[-ntest:,...], dtype=torch.float32)

train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test, y_test), batch_size=batch_size, shuffle=False)

# Select sample 10 from the test set
sample_index = 10

# Extract the input and output (ground truth) for the selected sample
x_sample = x_test[sample_index].cpu().numpy().squeeze()  # Input shape (1024, 2)
y_sample = y_test[sample_index].cpu().numpy().squeeze()  # Output shape (1024,)

# Print the input and output dimensions
print(f"Input shape (x): {x_sample.shape}")  # Shape (1024, 2)
print(f"Output shape (y): {y_sample.shape}")  # Shape (1024,)

# Plot both input channels in one figure
plt.figure(figsize=(10, 6))
plt.plot(x_sample[:, 1], x_sample[:, 0], label='u(x, t=0)', linestyle='--', color='orange')
plt.xlabel('Spatial Coordinate')
plt.ylabel('Value')
plt.legend()
plt.title(f'Input Channels for Sample {sample_index}')
plt.show()

# Plot the output (ground truth) in another figure
plt.figure(figsize=(10, 6))
plt.plot(x_sample[:, 1], y_sample, label='u(x, t=1)', marker='o', color='green')
plt.xlabel('Spatial Coordinate')
plt.ylabel('Value')
plt.legend()
plt.title(f'True Output for Sample {sample_index}')
plt.show()

4. Define Neural Operator Architecture

In [ ]:
class SpectralConv1d(nn.Module):
	"""docstring for SpectralConv1d"""
	def __init__(self, in_channels, out_channels, nModes):
		super(SpectralConv1d, self).__init__()
		self.in_channels = in_channels
		self.out_channels = out_channels
		self.nModes = nModes
		self.R = nn.Parameter(torch.rand(self.in_channels, self.out_channels, self.nModes, dtype = torch.cfloat))

	def forward(self, x):
		x_ft = torch.fft.rfft(x)
		x_ft_tr = x_ft[...,:self.nModes] # truncated
		# (batch, in_channel, x ), (in_channel, out_channel, x) -> (batch, out_channel, x)
		y_ft = torch.einsum('bix,iox->box', x_ft_tr, self.R)
		xx = torch.fft.irfft(y_ft, n=x.shape[-1])
		return xx

class FNO_1d(nn.Module):
	"""docstring for FNO_1d"""
	def __init__(self, nModes, width):
		super(FNO_1d, self).__init__()
		self.nModes = nModes
		self.width = width  #lifting channels P
		self.L = 128 #more lifing after Fourier layers
		self.nBlocks = 4

		self.fc0 = nn.Linear(2, self.width) # input channels = 2: (a(x),x)

		self.SConvs = nn.ModuleList([SpectralConv1d(self.width, self.width, self.nModes) for i in range(self.nBlocks)])
		self.LConvs = nn.ModuleList([nn.Conv1d(self.width, self.width, 1) for i in range(self.nBlocks)])

		self.fc1 = nn.Linear(self.width, self.L) #after Fourier layers
		self.fc2 = nn.Linear(self.L, 1) #reconstruction step: output_channels = 1

	def forward(self, x):
		x = self.fc0(x)
		x = x.permute(0, 2, 1)

		for i in range(self.nBlocks):
			x1 = self.SConvs[i](x)
			x2 = self.LConvs[i](x)
			x = x1 + x2
			if i < self.nBlocks - 1:
				x = F.gelu(x)

		x = x.permute(0, 2, 1)
		x = self.fc1(x)
		x = F.gelu(x)
		x = self.fc2(x)
		return x

5. Define the loss

In [ ]:
def normalized_loss(yhat, y):
	bs = yhat.shape[0]
	diff_norms = torch.norm(yhat.view(bs,-1) - y.view(bs,-1), 2, 1)
	y_norms = torch.norm(y.view(bs,-1),2,1)
	loss = torch.sum(diff_norms/y_norms)
	return loss

6. Perform training

In [ ]:
# Select device: CUDA if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Using device:', device)

# model (moved to device)
model = FNO_1d(modes, width).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

for ep in range(epochs):
    model.train()
    t1 = default_timer()
    train_l2 = 0
    for x, y in train_loader:
        # move batch to device
        x = x.to(device=device, dtype=torch.float32)
        y = y.to(device=device, dtype=torch.float32)

        optimizer.zero_grad()
        out = model(x)

        # use normalized relative loss (computed on device)
        l2 = normalized_loss(out, y)
        l2.backward()

        optimizer.step()
        train_l2 += l2.item()

    scheduler.step()
    train_l2 /= ntrain

    t2 = default_timer()
    if ep % 10 == 0 or ep == epochs - 1:
        print(f"Epoch {ep}: Time {t2-t1:.2f}s, Train L2 {train_l2:.6f}", flush=True)

7. Perform testing

In [ ]:
# Prediction / evaluation
# initialize prediction tensor on CPU
pred = torch.zeros(y_test.shape, dtype=torch.float32)
index = 0
# smaller batch size for prediction loop
test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test, y_test), batch_size=1, shuffle=False)

# Generate predictions
with torch.no_grad():
    for x, y in test_loader:
        test_l2 = 0
        # move inputs to device
        x_device = x.to(device=device, dtype=torch.float32)
        y_device = y.to(device=device, dtype=torch.float32)

        # get output on device
        out_device = model(x_device).view(-1)

        # accumulate prediction on CPU
        pred[index] = out_device.cpu()

        # compute loss on device and move scalar to CPU
        test_l2 += normalized_loss(out_device.view(1, -1), y_device.view(1, -1)).item()
        if index % 10 == 0:
            print(index, test_l2)
        index += 1

# save numpy predictions (ensure CPU numpy array)
np.save('./predictions.npy', pred.numpy())

# plotting (uses CPU arrays)
random_indices = np.random.choice(range(100), size=5, replace=False)

for ii in random_indices:
    x_values = x_test[ii, :, 1].cpu().numpy().squeeze()  # Use the second channel as x-axis

    plt.figure(figsize=(10, 6))

    plt.plot(x_values, pred[ii, :].numpy(), label='Prediction', linestyle='--', color='blue')
    plt.plot(x_values, y_test[ii, :].cpu().numpy(), label='Ground Truth', marker='o', color='green')

    plt.legend()
    plt.xlabel('Spatial Coordinate (x)')
    plt.ylabel('y')
    plt.title(f'Comparison of Test Data, Prediction, and Ground Truth for Sample {ii}')

    plt.show()